# 2 · Capa Plata: Actualización Incremental SCD Tipo 1 y SCD Tipo 2

Este notebook procesa de forma incremental los registros nuevos o modificados desde **`bronce_clientes`** hacia las tablas de Plata invocando los scripts desacoplados **`../sql/02_plata_merge_scd1.sql`** y **`../sql/02_plata_merge_scd2.sql`**.

* **Marca de agua dinámica (Watermark):** Calcula automáticamente la última fecha procesada (`MAX(fecha_cambio)` / `MAX(fecha_inicio)`) y deduplica por `id` con `QUALIFY`, eliminando filtros estáticos por número de entrega.
* **SCD Tipo 1 (`clientes_scd1`):** Mantiene la fotografía actual del cliente sobrescribiendo cambios.
* **SCD Tipo 2 (`clientes_scd2`):** Preserva la trazabilidad histórica cerrando versiones anteriores (`vigente = false`) y abriendo la nueva vigencia.

In [0]:
import sys
sys.path.append("../common")
from sql_runner import ejecutar_script_sql

# 1. Parámetros dinámicos (para ejecución manual o desde Databricks Jobs)
dbutils.widgets.text("catalogo", "workspace", "Catálogo")
dbutils.widgets.text("esquema", "demo_incremental", "Esquema")

catalogo = dbutils.widgets.get("catalogo")
esquema  = dbutils.widgets.get("esquema")

# 2. Ejecutar MERGE incremental SCD Tipo 1
df_scd1 = ejecutar_script_sql(
    spark=spark,
    ruta_sql="../sql/02_plata_merge_scd1.sql",
    catalogo=catalogo,
    esquema=esquema
)
print("✅ SCD Tipo 1 (clientes_scd1) actualizado:")
display(df_scd1)

# 3. Ejecutar MERGE + INSERT incremental SCD Tipo 2
ejecutar_script_sql(
    spark=spark,
    ruta_sql="../sql/02_plata_merge_scd2.sql",
    catalogo=catalogo,
    esquema=esquema
)
print("✅ SCD Tipo 2 (clientes_scd2) actualizado:")
display(spark.table(f"{catalogo}.{esquema}.clientes_scd2").orderBy("id", "fecha_inicio"))